In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/data.csv", sep=";")
df.columns = df.columns.str.strip()
print(df.shape)
df.head()

(4424, 37)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [2]:
df.info()
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification                          4424

In [3]:
print(df["Target"].value_counts())
print((df["Target"].value_counts(normalize=True) * 100).round(1))

Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64
Target
Graduate    49.9
Dropout     32.1
Enrolled    17.9
Name: proportion, dtype: float64


In [4]:
print(list(df.columns))

['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Previous qualification (grade)', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Admission grade', 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Unemployment rate', 'Inflation rate', 'GDP', 'Target']


In [5]:
SEED = 5659        
TARGET = "Target"

nominal_cols = [
    "Marital status", "Application mode", "Course", "Previous qualification",
    "Nacionality", "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
]
binary_cols = [
    "Daytime/evening attendance", "Displaced", "Educational special needs",
    "Debtor", "Tuition fees up to date", "Gender", "Scholarship holder",
    "International",
]
numeric_cols = [c for c in df.columns if c not in nominal_cols + binary_cols + [TARGET]]

print(len(nominal_cols), len(binary_cols), len(numeric_cols))
assert len(nominal_cols) + len(binary_cols) + len(numeric_cols) == 36

9 8 19


In [6]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

test_df = X_test.copy()
test_df[TARGET] = y_test
test_df.to_csv("../test_data.csv", index=False)

print("Train:", X_train.shape, " Test:", X_test.shape)
print(y_test.value_counts(normalize=True).mul(100).round(1))

Train: (3539, 36)  Test: (885, 36)
Target
Graduate    49.9
Dropout     32.1
Enrolled    18.0
Name: proportion, dtype: float64


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("nominal", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                  min_frequency=20, sparse_output=False), nominal_cols),
        ("binary", "passthrough", binary_cols),
        ("numeric", StandardScaler(), numeric_cols),
    ]
)

check = preprocessor.fit_transform(X_train)
print("Features after preprocessing:", check.shape[1])

Features after preprocessing: 119


In [8]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Baseline: untuned Gaussian NB, to show the one-hot problem
nb_default = Pipeline([("prep", clone(preprocessor)), ("clf", GaussianNB())])
nb_default.fit(X_train, y_train)
print(pd.crosstab(y_test, nb_default.predict(X_test),
                  rownames=["Actual"], colnames=["Predicted"]))

# Choose var_smoothing with 5-fold CV on the TRAINING data only
nb_search = GridSearchCV(
    Pipeline([("prep", clone(preprocessor)), ("clf", GaussianNB())]),
    param_grid={"clf__var_smoothing": np.logspace(-9, 0, 10)},
    scoring="f1_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
)
nb_search.fit(X_train, y_train)
best_vs = nb_search.best_params_["clf__var_smoothing"]
print("Best var_smoothing:", best_vs, "| CV macro-F1:", round(nb_search.best_score_, 4))

Predicted  Dropout  Enrolled  Graduate
Actual                                
Dropout        101       180         3
Enrolled        12       144         3
Graduate        53       368        21
Best var_smoothing: 0.1 | CV macro-F1: 0.6289


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=10,
                                            class_weight="balanced", random_state=SEED),
    "KNN": KNeighborsClassifier(n_neighbors=15, weights="distance"),
    "Naive Bayes": GaussianNB(var_smoothing=best_vs),
    "Support Vector Machine": CalibratedClassifierCV(
        SVC(kernel="rbf", C=1.0, class_weight="balanced"), ensemble=False
    ),
}

pipelines = {
    name: Pipeline([("prep", clone(preprocessor)), ("clf", model)])
    for name, model in classifiers.items()
}
print(list(pipelines))

['Logistic Regression', 'Decision Tree', 'KNN', 'Naive Bayes', 'Support Vector Machine']


In [10]:
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

rows = []
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)
    rows.append({
        "ML Model Name": name,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "AUC":       roc_auc_score(y_test, y_proba, multi_class="ovr",
                                   average="macro", labels=pipe.classes_),
        "Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall":    recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1":        f1_score(y_test, y_pred, average="macro", zero_division=0),
        "MCC":       matthews_corrcoef(y_test, y_pred),
    })
    print(f"Trained: {name}")

results = pd.DataFrame(rows).set_index("ML Model Name").round(4)
results

Trained: Logistic Regression
Trained: Decision Tree
Trained: KNN
Trained: Naive Bayes
Trained: Support Vector Machine


,Accuracy,AUC,Precision,Recall,F1,MCC
ML Model Name,,,,,,
Logistic Regression,0.7503,0.8909,0.7086,0.7108,0.7049,0.6036
Decision Tree,0.7006,0.8311,0.6822,0.6824,0.6689,0.5407
KNN,0.7073,0.8361,0.6689,0.5883,0.5977,0.5144
Naive Bayes,0.7096,0.8244,0.6633,0.6412,0.6460,0.5246
Support Vector Machine,0.7650,0.8795,0.7156,0.7094,0.7103,0.6191


In [11]:
import joblib
from pathlib import Path

model_dir = Path("../model")
model_dir.mkdir(exist_ok=True)

for name, pipe in pipelines.items():
    file_name = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(pipe, model_dir / file_name)
    print("Saved:", file_name)

results.to_csv(model_dir / "metrics_summary.csv")

Saved: logistic_regression.pkl
Saved: decision_tree.pkl
Saved: knn.pkl
Saved: naive_bayes.pkl
Saved: support_vector_machine.pkl
